This is my second attempt, where I scraped CELLxGENE Census github page for .md, .py and .ipynb files, combined them and attached to chatGPT https://chatgpt.com/share/670904ff-a098-8001-b522-56d7bb2bf25a

In [ ]:
# Installation commands are taken from: https://docs.scvi-tools.org/en/latest/tutorials/notebooks/hub/cellxgene_census_model.html
# See also: https://chanzuckerberg.github.io/cellxgene-census/cellxgene_census_docsite_installation.html
!pip install --quiet scvi-colab
!pip install --quiet cellxgene-census
!pip install --quiet pybiomart # is gene chrY-related?
from scvi_colab import install
install()
!pip install owlready2 # ontology processing

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.8/54.8 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.5/17.5 MB 44.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.0/129.0 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.4/77.4 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.5/49.5 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 35.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.8/16.8 MB 43.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 54.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.9/56.9 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.7/85.7 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.8/79.8 kB 2.6 MB/s eta 0:00:00
ERROR: pip's depe

In [ ]:
# Import necessary libraries
import cellxgene_census
import scanpy as sc
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import seaborn as sns
import scvi
from owlready2 import *

In [ ]:
# Select the CELLxGENE Census snapshot, see detals of dataset's scheme here:
#   https://raw.githubusercontent.com/chanzuckerberg/cellxgene-census/refs/heads/main/docs/cellxgene_census_schema.md
#   https://raw.githubusercontent.com/chanzuckerberg/single-cell-curation/refs/heads/main/schema/5.0.0/schema.md
census_version = "2024-07-01"
census = cellxgene_census.open_soma(census_version=census_version)

# Define what do we extract from Census
emb_names = [
    "scvi", # scVI integrated-embeddings with explicit modeling of batch effects
            # https://chanzuckerberg.github.io/cellxgene-census/notebooks/analysis_demo/comp_bio_scvi_model_use.html
    "geneformer", # Geneformer embeddings fine-tuned on CELLxGENE Census for multi-task learning
                  # https://chanzuckerberg.github.io/cellxgene-census/notebooks/analysis_demo/comp_bio_geneformer_prediction.html
    "scgpt" # scGPT: Towards Building a Foundation Model for Single-Cell Multi-omics Using Generative AI
            # http://dx.doi.org/10.1101/2023.04.30.538439
]  # from https://cellxgene.cziscience.com/census-models
organism = "homo_sapiens"
cell_type_counts_min = 5 # drop cells if the corresponding cell type has too few cells in a particular donor
cell_type_donors_min = 5 # drop cells if too few donors have the corresponding cell type

In [ ]:
# Download metadata of all primary (non-duplicated) cells from male donors for futher filtering
# Ontology ID is useful to have if the plain text name is ambigous
cell_metadata = cellxgene_census.get_obs(
    census,
    organism,
    value_filter = "sex == 'male' and is_primary_data == True",
    column_names = [
        'cell_type_ontology_term_id', 'cell_type',
        'development_stage_ontology_term_id', 'development_stage',
        'disease_ontology_term_id', 'disease',
        'donor_id',
        'soma_joinid'
    ]
)

In [ ]:
# cell_metadata_old = cell_metadata.copy()  # save
# cell_metadata = cell_metadata_old.copy()  # reload to avoid re-downloading

In [ ]:
# Only warn if missing data is actually present, print columns with missing data, and remove missing values
missing_data = cell_metadata.isna().sum()
missing_columns = missing_data[missing_data > 0]
if not missing_columns.empty:
    print("Warning: Missing data found in the following columns:")
    print(missing_columns)
cell_metadata = cell_metadata.dropna(subset=['cell_type', 'donor_id', 'development_stage', 'disease'])
del missing_data, missing_columns

In [ ]:
len(cell_metadata)

22371389

In [ ]:
# Working with HsapDv to get the donors age
onto = get_ontology("http://purl.obolibrary.org/obo/hsapdv.owl").load()

def HsapDv_to_age(ontology_id):
  term = onto.search_one(id=ontology_id)
  if term is None:
    return (None, None)

  # Extract start and end years from the term's attributes
  start_ypb = term.start_ypb[0] if term.start_ypb else None
  end_ypb = term.end_ypb[0] if term.end_ypb else None
  return (start_ypb, end_ypb)

In [ ]:
development_stages = cell_metadata.value_counts(subset=['development_stage_ontology_term_id', 'development_stage']).reset_index(name='count')
development_stages[['start_ypb', 'end_ypb']] = pd.DataFrame([HsapDv_to_age(x) for x in development_stages['development_stage_ontology_term_id']], columns=['start_ypb', 'end_ypb'])
max_ypb = development_stages['end_ypb'].max(skipna=True)
development_stages.loc[(development_stages['start_ypb'] > 1) & (development_stages['end_ypb'].isna()), 'end_ypb'] = max_ypb
development_stages = development_stages.sort_values(by=['start_ypb', 'end_ypb'], na_position='first')

# Show all values of `development_stage`
#pd.set_option('display.max_rows', None)
#pd.set_option('display.max_columns', None)
#pd.set_option('display.width', None)
#development_stages

In [ ]:
age_filter = development_stages['development_stage_ontology_term_id'][development_stages['start_ypb'] >= 40]
cell_metadata = cell_metadata[cell_metadata['development_stage_ontology_term_id'].isin(age_filter)]
del onto, development_stages, age_filter

In [ ]:
len(cell_metadata)

12741047

In [ ]:
# Select the Monocyte-related cell types and show their abundances
cell_metadata = cell_metadata.query("cell_type.str.contains('monocyte', case=False, na=False)")
print( cell_metadata.value_counts(subset=['cell_type_ontology_term_id', 'cell_type']).sort_values(ascending=False) )

cell_type_ontology_term_id  cell_type                                      
CL:0000860                  classical monocyte                                 284246
CL:0001054                  CD14-positive monocyte                             144239
CL:0002057                  CD14-positive, CD16-negative classical monocyte    132583
CL:0000576                  monocyte                                            77062
CL:0000875                  non-classical monocyte                              45475
CL:0002396                  CD14-low, CD16-positive monocyte                    19507
CL:0002397                  CD14-positive, CD16-positive monocyte               10895
CL:0002393                  intermediate monocyte                                2396
CL:0002470                  MHC-II-positive classical monocyte                    595
Name: count, dtype: int64


In [ ]:
len(cell_metadata)

716998

In [ ]:
# Because we have some follow up experiments, same 'donor_id' can have different 'disease' and/or 'development_stage'
cell_metadata['donor_follow_up'] = (
    cell_metadata['donor_id'].astype(str) + '_' +
    cell_metadata['disease'].astype(str) + '_' +
    cell_metadata['development_stage'].astype(str)
)

In [ ]:
# First, drop cells if the corresponding cell type has too few cells in the corresponding donor_follow_up
cell_metadata.loc[:, 'cell_type_size'] = cell_metadata.groupby(['cell_type', 'donor_follow_up'], observed=False).transform('size')
cell_metadata = cell_metadata[cell_metadata['cell_type_size'] >= cell_type_counts_min]
# cell_metadata[['cell_type', 'donor_follow_up', 'cell_type_size']].sort_values(by=['cell_type', 'donor_follow_up'])

In [ ]:
# Next, drop cells if too few donors still have the corresponding cell type
cell_metadata.loc[:, 'cell_type_donors'] = cell_metadata.groupby('cell_type', observed=False)['donor_follow_up'].transform('nunique')
cell_metadata = cell_metadata[cell_metadata['cell_type_donors'] >= cell_type_donors_min]
# cell_metadata[['cell_type', 'donor_follow_up', 'cell_type_size', 'cell_type_donors']].sort_values(by=['cell_type', 'donor_follow_up'])

In [ ]:
cell_metadata.value_counts(subset=['cell_type', 'donor_id', 'development_stage', 'disease']).sort_values()

cell_type                                        donor_id    development_stage          disease 
MHC-II-positive classical monocyte               PD44967     eighth decade human stage  normal          5
CD14-low, CD16-positive monocyte                 197_198     60-year-old human stage    normal          5
CD14-positive monocyte                           326_327     66-year-old human stage    normal          5
CD14-low, CD16-positive monocyte                 559_560     77-year-old human stage    normal          5
non-classical monocyte                           HGR0000092  45-year-old human stage    COVID-19        5
                                                                                                    ...  
classical monocyte                               Rep_C_1143  74-year-old human stage    COVID-19     7600
                                                 HGR0000051  55-year-old human stage    COVID-19     8590
                                                 D496        sixth decade human stage   normal       9744
CD14-positive, CD16-negative classical monocyte  P-S085      62-year-old human stage    COVID-19    10617
monocyte                                         H06         72-year-old human stage    normal      16088
Name: count, Length: 2071, dtype: int64

In [ ]:
len(cell_metadata)

716343

In [ ]:
# Because we have some follow up experiments, same 'donor_id' can have different 'disease' and/or 'development_stage'
# But there could also be mistakes, lets check we do not have them

cell_metadata.loc[:,'n_disease_per_donor'] = cell_metadata.groupby(['donor_id'], observed=False)['disease'].transform('nunique')
cell_metadata.loc[:,'n_development_stage_per_donor'] = cell_metadata.groupby(['donor_id'], observed=False)['development_stage'].transform('nunique')
filtered_data = cell_metadata[(cell_metadata['n_disease_per_donor'] > 1) | (cell_metadata['n_development_stage_per_donor'] > 1)]
#filtered_data = filtered_data.sort_values(by=['donor_id', 'disease', 'development_stage'])
filtered_data[['donor_id', 'development_stage', 'disease']].value_counts(sort=False)

donor_id  development_stage           disease                          
AP1       56-year-old human stage     COVID-19                              664
                                      post-COVID-19 disorder                797
AP2       66-year-old human stage     COVID-19                              265
                                      post-COVID-19 disorder                230
H2        49-year-old human stage     normal                                401
          sixth decade human stage    normal                                 68
H4        49-year-old human stage     normal                                368
          sixth decade human stage    normal                                 62
PD43824   fifth decade human stage    nonpapillary renal cell carcinoma     839
                                      normal                                870
PD44966   sixth decade human stage    nonpapillary renal cell carcinoma      40
                                      normal                               2966
PD44967   eighth decade human stage   nonpapillary renal cell carcinoma      15
                                      normal                               2444
PD45814   seventh decade human stage  nonpapillary renal cell carcinoma     551
                                      normal                                233
PD45815   sixth decade human stage    nonpapillary renal cell carcinoma     233
                                      normal                               1330
PD47172   sixth decade human stage    kidney oncocytoma                     279
                                      normal                                391
PD47512   sixth decade human stage    nonpapillary renal cell carcinoma      68
                                      normal                               4757
Name: count, dtype: int64

In [ ]:
# Subsample 1e5 random cells to avoid crashing Colab's RAM if we have a lot of cells
if len(cell_metadata) > 100000:
    cell_metadata = cell_metadata.sample(n=100000, random_state=1)

In [ ]:
# Download scRNA-seq data
obs_coords_ids = cell_metadata["soma_joinid"].tolist()
adata = cellxgene_census.get_anndata(
  census,
  organism=organism,
  measurement_name="RNA",
  obs_coords=obs_coords_ids,
  obs_embeddings=emb_names
)

adata

AnnData object with n_obs × n_vars = 100000 × 60530
    obs: 'soma_joinid', 'dataset_id', 'assay', 'assay_ontology_term_id', 'cell_type', 'cell_type_ontology_term_id', 'development_stage', 'development_stage_ontology_term_id', 'disease', 'disease_ontology_term_id', 'donor_id', 'is_primary_data', 'observation_joinid', 'self_reported_ethnicity', 'self_reported_ethnicity_ontology_term_id', 'sex', 'sex_ontology_term_id', 'suspension_type', 'tissue', 'tissue_ontology_term_id', 'tissue_type', 'tissue_general', 'tissue_general_ontology_term_id', 'raw_sum', 'nnz', 'raw_mean_nnz', 'raw_variance_nnz', 'n_measured_vars'
    var: 'soma_joinid', 'feature_id', 'feature_name', 'feature_length', 'nnz', 'n_measured_obs'
    obsm: 'scvi', 'geneformer', 'scgpt'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

adata.write('/content/drive/My Drive/DDLS2024/Monocytes-like_male_1e5cells.h5ad')

Mounted at /content/drive


In [ ]:
census.close()
del census